# 第9回 トレンドと移動平均


第8回では、ドル・円レートの値に AR(1) モデルを当てはめると傾きがほぼ1になり、次の日の予測には過去の平均より今日の値がよいことを確かめた。

この回では、レートの長い期間の動きを考える。ドル・円は2000年から見て円安に向かっているのだろうか。見る期間を変えると答えはどう変わり、その答えは次の日の予測に使えるだろうか。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下の2つのセルを実行する。2つ目のセルは `statsmodels` を読み込む。


In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


In [ ]:
import sys
if sys.platform == "emscripten":
    import pyodide_js
    await pyodide_js.loadPackage("statsmodels")
import statsmodels.api as sm


## 第8回 演習1の解説

第8回の演習1は、レートの値に AR(1) モデルを当てはめると傾きがほぼ1になること、そのような系列では過去の平均が予測の役に立たないことを確かめる問いだった。

### 問1：ユーロ・ドルのレートの値

> 問1の計算欄のコードは、FRBのニューヨーク正午のユーロ・ドル（ドル／ユーロ）のレートの値に AR(1) モデルを当てはめ、傾きと 1 − 5.3 ÷ 日数の値を表示する。2024年の各日のレートを2000〜2023年の平均で予想した場合と、今日の値で予想した場合の誤差の大きさの平均も表示する。セルを実行し、4つの値を答えよ。また、ユーロ・ドルのレートの予想に過去の平均が役に立つかを答えよ。


In [ ]:
boj = load_boj_fx()
rate = boj["usd_jpy"].dropna()
fed = load_fed_fx()

eur_usd = fed["eur_usd"].dropna()
eur_levels = pd.DataFrame({
    "前日の値": eur_usd.shift(1),
    "当日の値": eur_usd,
}).dropna()
eur_train = eur_levels.loc[:"2023"]
model_eur = sm.OLS(eur_train["当日の値"], sm.add_constant(eur_train["前日の値"])).fit()
eur_check = eur_levels.loc["2024"]
eur_error_mean = (eur_check["当日の値"] - eur_usd.loc[:"2023"].mean()).abs().mean()
eur_error_naive = (eur_check["当日の値"] - eur_check["前日の値"]).abs().mean()
print(f"傾き φ：{model_eur.params['前日の値']:.4f}（{len(eur_train)} 日）")
print(f"1 − 5.3 ÷ 日数：{1 - 5.3 / len(eur_train):.4f}")
print(f"平均による予測の平均絶対誤差（2024年）：{eur_error_mean:.4f} ドル／ユーロ")
print(f"今日の値による予測の平均絶対誤差（2024年）：{eur_error_naive:.4f} ドル／ユーロ")


傾きは0.9989で、1にきわめて近い。ランダムウォークのように傾きが本当に1の系列でも、最小二乗法で求めた傾きは1より少し小さく出る。5.3 ÷ 日数は、そのずれのおよその大きさで、日数が多いほどずれは小さくなる。2000〜2023年の6016日では 1 − 5.3 ÷ 6016 = 0.9991 で、求めた傾き0.9989とほぼ同じである。ユーロ・ドルのレートも、傾きが1の系列と見分けがつかない。

2024年の平均絶対誤差は、2000〜2023年の平均による予測が0.1119ドル／ユーロ、今日の値による予測が0.0029ドル／ユーロで、平均による予測は38倍ほど大きい。ユーロ・ドルでも、過去の平均は予測の役に立たない。


### 問2：3つの文の正誤

(a) と問3では、ドル・円レートが2000〜2023年の平均へ戻ったかを使う。平均と、2022〜2025年の最も低い値、2024年の2つの予測の平均絶対誤差を表示しよう。


In [ ]:
levels = pd.DataFrame({
    "前日の値": rate.shift(1),
    "当日の値": rate,
}).dropna()
past_mean = rate.loc[:"2023"].mean()
test_level = levels.loc["2024"]
error_mean = (test_level["当日の値"] - past_mean).abs().mean()
error_naive = (test_level["当日の値"] - test_level["前日の値"]).abs().mean()
print(f"2000〜2023年の平均：{past_mean:.1f} 円／ドル")
print(f"2022〜2025年の最も低い値：{rate.loc['2022':'2025'].min():.2f} 円／ドル")
print(f"平均による予測の平均絶対誤差（2024年）：{error_mean:.2f} 円／ドル")
print(f"今日の値による予測の平均絶対誤差（2024年）：{error_naive:.2f} 円／ドル")


(a) 「ドル・円レートに当てはめた AR(1) モデルの傾きは0.9994で1より小さいので、レートはいずれ2000〜2023年の平均の109.2円へ戻っていく。」

誤りである。2022〜2025年の最も低い値は113.76円で、平均の109.2円よりはるかに上る。傾きが1より少し小さいくらいでは平均へ戻らない。

(b) 「ドリフトのないランダムウォークでは、期間が長くなるほど値の散らばりが大きくなる。」

正しい。$t$ 期の値は $t$ 個の乱数の合計なので、その標準偏差は1期分の $\sqrt{t}$ 倍になる。

(c) 「ドリフトのないランダムウォークで上昇が長く続いたあとは、次の期も上がると予想できる。」

誤りである。次の期の値は、今の値に平均0の乱数 $\varepsilon_t$ を足した値で、$\varepsilon_t$ はそれまでの上昇と関係なく決まる。上昇が続いたあとでも次の期に上がるか下がるかは分からず、予測は今の値になる。


### 問3：Cさんの考え

> Cさんは「ドル・円レートの2000〜2023年の平均は109.2円だった。今は150円台なので、いずれ109円台へ戻るはずだ。今のうちにドルを売って円にしておけば得をする」と考えた。この回の結果を使って、Cさんの考えの問題点を3〜5文で説明せよ。

解答には、平均へ戻る動きが見られないことと、平均による予測の平均絶対誤差が入っているとよい。解答の例は次のとおり。

> Cさんは、レートが2000〜2023年の平均の109.2円へ戻ると考えている。しかし2022〜2025年のレートは最も低い日でも113.76円で、4年間ずっと平均より上にあった。2024年の各日を平均で予測すると平均絶対誤差は42.42円で、今日の値による予測の0.70円の60倍を超える。平均へ戻ることを当てにしてドルを売っても、得をするとはいえない。


### 追加問題：AR(1) モデルの予測

> 追加問題：本文で求めたドル・円レートの AR(1) モデル（切片 `c_level`、傾き `phi_level`）で2024年の各日のレートを予想し、誤差の大きさの平均を求めよ。今日の値をそのまま使う予想の誤差の大きさの平均と比べ、どちらの予想がよいかを答えよ。

まず、第8回の本文と同じく、2000〜2023年の前日の値と当日の値から切片と傾きを求める。`levels` は問2のセルで作った、前日の値と当日の値を並べた表である。


In [ ]:
train_level = levels.loc[:"2023"]
model_level = sm.OLS(train_level["当日の値"], sm.add_constant(train_level["前日の値"])).fit()
c_level = model_level.params["const"]
phi_level = model_level.params["前日の値"]
print(f"切片 c：{c_level:.3f}、傾き φ：{phi_level:.4f}")


2024年の各日の予測は、切片 + 傾き × 前日の値 で求める。最初の5日を、前日の値と当日の値と並べて表示する。


In [ ]:
pred_level = c_level + phi_level * test_level["前日の値"]
pd.DataFrame({
    "前日の値": test_level["前日の値"],
    "AR(1) モデルの予測": pred_level,
    "当日の値": test_level["当日の値"],
}).head()


予測は前日の値とほとんど同じになる。最後に、当日の値と予測の差の絶対値を平均して、平均絶対誤差を求める。


In [ ]:
error_ar = (test_level["当日の値"] - pred_level).abs().mean()
print(f"AR(1) モデルによる予測：{error_ar:.3f} 円／ドル")
print(f"今日の値による予測：{error_naive:.3f} 円／ドル")


AR(1) モデルによる予測の平均絶対誤差は0.699円で、今日の値による予測の0.696円とほぼ同じである。傾きがほぼ1なので、AR(1) モデルの予測は今日の値とほとんど変わらない。この回でも、次の日の予測を比べる相手には今日の値を使う。


## ドル・円のトレンド

長い期間にわたって値が上がり続けたり下がり続けたりする動きを、トレンドという。トレンドは直線とは限らず、上がる向きから下がる向きへ変わることもある。2000〜2025年のドル・円レートを図にして、どんなトレンドがあるかを見よう。最も低い日と最も高い日は、第3回で使った `.idxmin()` と `.idxmax()` で求める。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.index, rate, linewidth=0.8)
ax.set(title="ドル・円レート（東京17時、2000〜2025年）", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()

print(f"最初の日：{rate.index[0].date()}、{rate.iloc[0]:.2f} 円／ドル")
print(f"最も低い日：{rate.idxmin().date()}、{rate.min():.2f} 円／ドル")
print(f"最も高い日：{rate.idxmax().date()}、{rate.max():.2f} 円／ドル")
print(f"最後の日：{rate.index[-1].date()}、{rate.iloc[-1]:.2f} 円／ドル")


レートは2000年1月4日の102.99円から2002年にかけて上がったあと、上下しながら下がり、2011年10月28日には75.84円になった（円高）。その後は上がって、2024年7月3日に161.77円になった（円安）。最初の日と最後の日（2025年12月30日、155.98円）を比べると53円ほどの円安だが、その間には下がる時期と上がる時期が入れ替わっている。トレンドの向きは、どの期間を見るかで変わりそうである。


## 線形トレンド

トレンドを1つの数で表すために、レートに時間の1次式を当てはめる。$t$ 日目のレートを $y_t$、最初の日からの経過年数を $x_t$ として、

$$y_t = a + b\, x_t + u_t$$

の切片 $a$ と傾き $b$ を最小二乗法で求める。$a + b\, x_t$ のように時間の1次式で表したトレンドを線形トレンドといい、図では直線になる。$u_t$ は線形トレンドからのずれである。$b$ は経過年数が1年増えるごとに線形トレンドが何円動くかを表し、正なら1年あたりの円安、負なら円高の大きさになる。第7回では前の期の値を説明変数にして `sm.OLS` で回帰したが、ここでは経過年数を説明変数にする。

経過年数は、各日の日付から最初の日の日付を引いて作る。`rate.index - rate.index[0]` は各日が最初の日から何日後かを表す列で、`.days` を付けるとその日数を数として取り出せる。日数を365.25で割ると年になる。365.25は、4年に1度のうるう年を含めて平均した1年の日数である。`pd.Series` の `name="経過年数"` は列に名前を付ける指定で、回帰の結果から傾きをこの名前で取り出せるようにする。


In [ ]:
years = pd.Series((rate.index - rate.index[0]).days / 365.25, index=rate.index, name="経過年数")
print(f"最後の日の経過年数：{years.iloc[-1]:.2f} 年")
pd.DataFrame({"レート": rate, "経過年数": years}).head()


最初の日の経過年数は0、次の日は1日後なので 1 ÷ 365.25 = 0.003 になる。最後の日の2025年12月30日は25.99年後である。休日にはレートがないので表にも行がないが、経過年数は日付から計算するので、休日をはさんでも日付の間隔どおりの値になる。

経過年数を説明変数にして、2000〜2025年の全期間で線形トレンドを求めよう。`model_all.params` の `"const"` が切片、`"経過年数"` が傾きである。求めた線形トレンドの値 $a + b\, x_t$ を、レートと重ねて描く。


In [ ]:
model_all = sm.OLS(rate, sm.add_constant(years)).fit()
a_all = model_all.params["const"]
b_all = model_all.params["経過年数"]
line_all = a_all + b_all * years
print(f"切片 a：{a_all:.2f} 円／ドル、傾き b：{b_all:.3f} 円／ドル／年")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.index, rate, linewidth=0.8, label="ドル・円レート")
ax.plot(rate.index, line_all, color="black", linewidth=1.5, label="2000〜2025年の線形トレンド")
ax.set(title="ドル・円レートと線形トレンド", ylabel="円／ドル")
ax.legend()
format_axis(ax)
fig.tight_layout()
plt.show()


傾きは1年あたり0.889円で、線形トレンドは26年間で23円ほどしか上がらない。最初の日と最後の日の差の53円より、かなり小さい。図の線形トレンドはレートの真ん中を通っているが、レートは2000年代の前半には線形トレンドより上に、2009〜2013年には下に、2022〜2025年には上にあり、線形トレンドから長い間離れている。1本の直線では、途中で向きが変わる動きを表せない。


## 期間で変わる傾き

図では、レートは上下しながら2011年まで下がり、2012〜2015年に上がり、2016〜2020年はほぼ横ばいで、2021年から再び上がっている。この4つの期間に分けて、それぞれに線形トレンドを当てはめよう。`starts` と `ends` に期間の最初と最後の年を並べ、`for` の繰り返しで期間ごとに `sm.OLS` を使う。`years.loc[part.index]` は、その期間の日の経過年数を取り出す。


In [ ]:
starts = ["2000", "2012", "2016", "2021"]
ends = ["2011", "2015", "2020", "2024"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.index, rate, color="lightgray", linewidth=0.8, label="ドル・円レート")
ax.plot(rate.index, line_all, color="black", linestyle="--", linewidth=1, label="2000〜2025年（全期間）")
for k in range(4):
    part = rate.loc[starts[k]:ends[k]]
    model_part = sm.OLS(part, sm.add_constant(years.loc[part.index])).fit()
    b_part = model_part.params["経過年数"]
    line_part = model_part.params["const"] + b_part * years.loc[part.index]
    ax.plot(part.index, line_part, linewidth=2, label=f"{starts[k]}〜{ends[k]}年")
    print(f"{starts[k]}〜{ends[k]}年：{len(part)} 日、傾き {b_part:.3f} 円／ドル／年")
ax.set(title="期間ごとの線形トレンド", ylabel="円／ドル")
ax.legend(fontsize=9, loc="lower right", ncol=2)
format_axis(ax)
fig.tight_layout()
plt.show()


傾きは、2000〜2011年が −3.009 円、2012〜2015年が12.872円、2016〜2020年が −0.790 円、2021〜2024年が13.584円（いずれも1年あたり）で、期間によって符号も大きさも変わる。全期間の0.889円は、円高が続いた期間と円安が続いた期間をまとめて1本の線形トレンドにした値である。「ドル・円は円安に向かっているか」という問いの答えは、2000〜2011年を見れば1年に3円ほどの円高、2021〜2024年を見れば1年に13円を超える円安となり、見る期間で変わる。

期間の区切りは、図を見て向きが変わったところで決めた。区切りを増やせば、期間ごとの線形トレンドはレートによく合うようになるが、区切りの選び方しだいで傾きは変わる。過去のレートによく合う線形トレンドがこれからの予測に役立つかは、別に確かめる必要がある。


## 線形トレンドを延ばした予測

線形トレンドを2024年まで延ばし、その日の線形トレンドの値を2024年の各日のレートの予測にしてみよう。線形トレンドは2023年までのレートだけで求める。2000〜2023年の線形トレンドと、直近の2021〜2023年の線形トレンドを求め、第5回から比べる相手にしてきた今日の値による予測と、2024年の平均絶対誤差を比べる。


In [ ]:
part = rate.loc["2000":"2023"]
model_long = sm.OLS(part, sm.add_constant(years.loc[part.index])).fit()
line_long = model_long.params["const"] + model_long.params["経過年数"] * years

part = rate.loc["2021":"2023"]
model_short = sm.OLS(part, sm.add_constant(years.loc[part.index])).fit()
line_short = model_short.params["const"] + model_short.params["経過年数"] * years

test = rate.loc["2024"]
print(f"2000〜2023年の線形トレンド（傾き {model_long.params['経過年数']:.3f}）：{(test - line_long.loc['2024']).abs().mean():.3f} 円／ドル")
print(f"2021〜2023年の線形トレンド（傾き {model_short.params['経過年数']:.3f}）：{(test - line_short.loc['2024']).abs().mean():.3f} 円／ドル")
print(f"今日の値による予測：{(test - rate.shift(1).loc['2024']).abs().mean():.3f} 円／ドル")

recent = rate.loc["2021":]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(recent.index, recent, linewidth=0.8, label="ドル・円レート")
ax.plot(recent.index, line_long.loc["2021":], color="black", linewidth=1.5, label="2000〜2023年の線形トレンド")
ax.plot(recent.index, line_short.loc["2021":], color="darkred", linewidth=1.5, label="2021〜2023年の線形トレンド")
ax.axvline(test.index[0], color="gray", linestyle=":")
ax.set(title="2023年までの線形トレンドを2025年まで延ばす", ylabel="円／ドル")
ax.legend()
format_axis(ax)
fig.tight_layout()
plt.show()


2000〜2023年の線形トレンドによる予測は、平均絶対誤差が39.018円で、今日の値による予測の0.696円の50倍を超える。この線形トレンドの傾きは1年あたり0.272円とほぼ横ばいで、図の黒い線のとおり2024年も110円台にとどまり、140円台から160円台で動いたレートから大きく離れた。2021〜2023年の線形トレンドは7.892円で、2000〜2023年の線形トレンドよりずっと小さいが、それでも今日の値による予測の10倍を超える。

図の点線より右が、線形トレンドを求めるのに使っていない期間である。2021〜2023年の線形トレンドは2024年前半のレートに近いが、レートが2024年7月から下がったあとも上がり続け、2025年にはレートとの差が開いていく。線形トレンドを延ばした予測は、線形トレンドを求めた期間の傾きがその後も続くと考えている。線形トレンドを求めた期間によく合う線形トレンドでも、その後の期間の予測が当たるとは限らない。


## 移動平均

線形トレンドの代わりに、レートの動きに沿って曲がる線でトレンドを表す方法がある。$t$ 日目とその前後 $k$ 日ずつ、合わせて $m = 2k + 1$ 日のレートの平均

$$\hat T_t = \frac{y_{t-k} + \cdots + y_{t} + \cdots + y_{t+k}}{m}$$

を、$m$ 日の中心移動平均という。近い日のレートは近い値をとりやすいので、平均すると日々の細かい上下がならされ、大きな動きが残る。前と後の日数をそろえるため、$m$ は奇数にする。

`rate.rolling(m, center=True)` は、各日を中心とする $m$ 日分のレートをまとめる。続けて `.mean()` を付けると、その平均が求まる。休日にはレートがないので、ここでの日数はレートのある日を数える。まず $m = 5$ で、2024年7月3日の値を確かめよう。


In [ ]:
ma5 = rate.rolling(5, center=True).mean()
print(rate.loc["2024-06-28":"2024-07-09"])
print(f"7月3日の5日の中心移動平均：{ma5.loc['2024-07-03']:.3f} 円／ドル")
print(f"7月1〜5日の5日の平均：{rate.loc['2024-07-01':'2024-07-05'].mean():.3f} 円／ドル")


7月3日の5日の中心移動平均161.298円は、7月3日とその前後2日ずつ（7月1日、2日、4日、5日）のレートの平均と同じである。7月5日の次にレートのある日は、週末をはさんだ7月8日になる。

平均する日数を増やすと、曲線はどう変わるだろうか。1年分と1か月分の平均にあたる日数を、2024年にレートのある日の数から決めよう。


In [ ]:
days_2024 = len(rate.loc["2024"])
print(f"2024年にレートのある日：{days_2024} 日、12で割ると {days_2024 / 12:.1f} 日")


2024年にレートのある日は245日で、1か月あたりでは20.4日になる。中心移動平均の日数は奇数にするので、1年分を245日、1か月分を20.4に近い奇数の21日とする。21日と245日の中心移動平均を、2022〜2025年について描こう。`.dropna()` で値のない日を除き、`.index[-1]` で曲線の最後の日を表示する。


In [ ]:
ma21 = rate.rolling(21, center=True).mean()
ma245 = rate.rolling(245, center=True).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.loc["2022":].index, rate.loc["2022":], color="lightgray", linewidth=0.8, label="ドル・円レート")
ax.plot(ma21.loc["2022":].index, ma21.loc["2022":], linewidth=1.5, label="21日の中心移動平均")
ax.plot(ma245.loc["2022":].index, ma245.loc["2022":], linewidth=2, label="245日の中心移動平均")
ax.set(title="中心移動平均（2022〜2025年）", ylabel="円／ドル")
ax.legend()
format_axis(ax)
fig.tight_layout()
plt.show()

print(f"21日の中心移動平均の最後の日：{ma21.dropna().index[-1].date()}")
print(f"245日の中心移動平均の最後の日：{ma245.dropna().index[-1].date()}")


21日の中心移動平均は数週間ごとの上下を残し、245日の中心移動平均は1年ほどの大きな上がり下がりだけを残している。平均する日数が多いほど、曲線は滑らかになる。

どちらの曲線も、データの最後の2025年12月30日まで届いていない。21日の曲線は2025年12月16日、245日の曲線は2025年7月2日で終わる。$m$ 日の中心移動平均には、その日のあとの $k$ 日分（21日なら10日、245日なら122日）のレートが要る。最後の $k$ 日には、そのあとのレートがまだないので計算できない。


## 後方移動平均

今日の時点では、明日からのレートはまだ分からない。今日までに分かっているレートだけで平均を求めるには、今日とその前の $m - 1$ 日の平均

$$M_t = \frac{y_{t-m+1} + \cdots + y_{t-1} + y_t}{m}$$

を使う。これを $m$ 日の後方移動平均という。`rate.rolling(m).mean()` のように `center=True` を付けずに計算すると、後方移動平均になる。

2つの移動平均の違いは、2024年6月末までのレートしか分からない時点に立つとはっきりする。`known` に6月末までのレートを入れ、6月の最後の5日について、全期間のレートで計算した値と比べよう。


In [ ]:
known = rate.loc[:"2024-06"]
pd.DataFrame({
    "中心（全期間）": rate.rolling(21, center=True).mean(),
    "中心（6月末まで）": known.rolling(21, center=True).mean(),
    "後方（全期間）": rate.rolling(21).mean(),
    "後方（6月末まで）": known.rolling(21).mean(),
}).loc["2024-06-24":"2024-06-28"]


表はどれも21日の移動平均である。6月末までのレートだけで計算すると、中心移動平均は6月24〜28日の値がない（NaN）。この5日の中心移動平均には、7月のレートが使われているからである。後方移動平均は、6月末までのレートで計算しても全期間のレートで計算しても同じ値になる。その日までに分かっていたレートだけで計算できるのは、後方移動平均である。

そのかわり、後方移動平均は動きが遅れる。2024年7月3日の最高値の前後（2024年1月〜2025年6月）で、21日と245日の後方移動平均と、245日の中心移動平均を比べよう。4つの列を表 `compare` にまとめ、`for` の繰り返しで列ごとに線を描き、最も高くなった日を `.idxmax()` で表示する。


In [ ]:
compare = pd.DataFrame({
    "ドル・円レート": rate,
    "21日の後方移動平均": rate.rolling(21).mean(),
    "245日の後方移動平均": rate.rolling(245).mean(),
    "245日の中心移動平均": ma245,
}).loc["2024-01":"2025-06"]

fig, ax = plt.subplots(figsize=(9, 4))
for name in compare.columns:
    ax.plot(compare.index, compare[name], linewidth=1.5, label=name)
    print(f"{name}が最も高い日：{compare[name].idxmax().date()}")
ax.set(title="2024年7月の最高値の前後", ylabel="円／ドル")
ax.legend(fontsize=9)
format_axis(ax)
fig.tight_layout()
plt.show()


レートが最も高かった2024年7月3日に対して、21日の後方移動平均は7月16日、245日の後方移動平均は2025年2月21日に最も高くなった。後方移動平均は今日までの日だけを平均するので、レートが下がり始めても、平均に残っている高い日の分だけ遅れて動く。平均する日数が多いほど遅れは大きく、245日では7か月以上遅れた。245日の中心移動平均は、前と後の日を同じ数だけ使うので、2024年8月21日に最も高くなり、遅れは小さい。


## 移動平均による次の日の予測

後方移動平均は今日までのレートで計算できるので、明日の予測に使える。今日までの $m$ 日の平均を、明日の予測にする。演習1の追加問題でも同じ計算をするので、$m = 5$ の場合を3つの段階に分けて確かめ、そのあと平均する日数を変えて比べよう。

### 前日までの5日の平均

`rate.rolling(5).mean()` は、各日の行にその日までの5日の平均を入れる。この平均を予測に使うのは次の日なので、第5回で使った `.shift(1)` で1行下にずらし、各日の行に前日までの5日の平均を入れる。当日の値と2つの平均を表 `steps` に並べ、2024年1月4〜12日を表示しよう。


In [ ]:
steps = pd.DataFrame({
    "当日の値": rate,
    "その日までの5日の平均": rate.rolling(5).mean(),
    "予測（前日までの5日の平均）": rate.rolling(5).mean().shift(1),
})
print(f"1月4〜11日の5日の平均：{rate.loc['2024-01-04':'2024-01-11'].mean():.3f} 円／ドル")
steps.loc["2024-01-04":"2024-01-12"]


1月12日の予測の144.572円は、1行上の1月11日の「その日までの5日の平均」と同じ値である。1月11日までの5日は1月4日、5日、9日、10日、11日で、成人の日の1月8日はレートがないので数えない。この5日の平均は、表の上に表示した144.572円と一致する。2024年の最初の日の1月4日の予測141.954円には、2023年の最後の5日のレートが使われている。

### 誤差と絶対誤差

第5回と同じように、当日の値から予測を引いた差を誤差とし、`.abs()` で符号を外して絶対誤差にする。2つの列を `steps` に足そう。


In [ ]:
steps["誤差"] = steps["当日の値"] - steps["予測（前日までの5日の平均）"]
steps["絶対誤差"] = steps["誤差"].abs()
steps.loc["2024-01-04":"2024-01-12"]


この期間のレートは143円台から145円台へ上がり、前日までの5日の平均は当日の値より低かったので、誤差はどの日もプラスになった。レートが上がっている途中では、平均に残っている前の低い日の分だけ予測が低くなる。前の節で見た後方移動平均の遅れが、予測の誤差になって表れている。

### 2024年の平均絶対誤差

`steps.loc["2024"]["絶対誤差"]` で2024年の絶対誤差を取り出し、`.mean()` で平均する。同じ値は、3つの段階を1行につないだ式でも求まる。式の中の `test` は、「線形トレンドを延ばした予測」の節で作った2024年のレートである。


In [ ]:
print(f"表 steps から：{steps.loc['2024']['絶対誤差'].mean():.3f} 円／ドル")
print(f"1行の式から：{(test - rate.rolling(5).mean().shift(1).loc['2024']).abs().mean():.3f} 円／ドル")


どちらも1.117円で、今日の値による予測の0.696円より大きい。1行の式は、内側から次の順に計算している。

| 式の部分 | 求めるもの | 表 `steps` の列 |
|---|---|---|
| `rate.rolling(5).mean()` | その日までの5日の平均 | その日までの5日の平均 |
| `.shift(1)` | 1行下にずらした、前日までの5日の平均 | 予測（前日までの5日の平均） |
| `.loc["2024"]` | 2024年の日の予測 | |
| `test - ( … )` | 当日の値から予測を引いた差 | 誤差 |
| `.abs()` | 符号を外した誤差 | 絶対誤差 |
| `.mean()` | 2024年の平均絶対誤差 | |

### 平均する日数を変えた比較

1行の式の5を $m$ に置き換えると、平均する日数を変えた予測の誤差が求まる。`for m in [1, 5, 21, 245]:` は、第3回で練習した `for` の繰り返しで、$m$ に1、5、21、245を順に入れて、その下の字下げした3行を4回実行する。3行では、1行の式を `forecast`（予測）と `error`（平均絶対誤差）の2つに分けて書き、`print` で表示する。$m = 1$ の平均は今日の値そのものなので、今日の値による予測も同じ式で計算できる。


In [ ]:
for m in [1, 5, 21, 245]:
    forecast = rate.rolling(m).mean().shift(1)
    error = (test - forecast.loc["2024"]).abs().mean()
    print(f"{m} 日の平均による予測：{error:.3f} 円／ドル")


平均絶対誤差は、1日（今日の値）が0.696円で最も小さく、5日で1.117円、21日で2.538円、245日で6.098円と、平均する日数が多いほど大きい。移動平均は日々の上下をならしてトレンドを見せるが、明日の予測としては、遅れの分だけ今日の値より誤差が大きくなる。

この回の問いに戻ろう。ドル・円のトレンドの向きは期間によって変わり、2000〜2025年の全期間の傾きは1年あたり0.889円とほぼ横ばいだった。2021〜2024年の傾きは13.584円の円安の向きだったが、線形トレンドを延ばした予測も移動平均による予測も、次の日の予測の誤差は今日の値より大きかった。トレンドは過去の動きをまとめるのに役立ち、次の日の予測には今日の値を使うのがよい。


## 演習1：トレンドと移動平均

1. 問1の計算欄のコードは、FRBのニューヨーク正午のユーロ・ドル（ドル／ユーロ）に線形トレンドを当てはめ、2000〜2025年の全期間と、2000〜2007年、2008〜2016年、2017〜2025年の3つの期間の傾きを表示する。2000〜2023年の線形トレンドを延ばした予測と今日の値による予測について、2024年の平均絶対誤差も表示する。セルを実行し、4つの傾きと2つの平均絶対誤差を答えよ。また、ユーロ・ドルの次の日の予測には2つのうちどちらを使うのがよいかを答えよ。
2. 問2の計算欄のコードは、本文で求めたドル・円の2021〜2023年の線形トレンドを延ばし、2025年12月30日と2026年12月30日の線形トレンドの値を表示する。2025年12月30日の実際のレートも表示する。セルを実行し、次の(a)と(b)に答えよ。
   - (a) 2026年12月30日のレートをこの線形トレンドで予測すると何円になるか。この予測はどんな前提を置いているかを1文で書け。
   - (b) 2025年12月30日の線形トレンドの値と実際のレートを比べ、(a) の前提が2025年に成り立ったかを1文で答えよ。
3. Hさんは来月の海外旅行に向けて、ドル・円を毎日見ている。Hさんが知りたいのは、①明日のレートの予測と、②この1年のレートの大まかな動きの2つである。今日の値、後方移動平均、中心移動平均のうち、①と②にはそれぞれどれを使うのがよいか。この回の結果を使って、3〜5文で説明せよ。

追加問題：「移動平均による次の日の予測」のセルを参考に、平均する日数を2日、3日、10日にして2024年の平均絶対誤差を求め、今日の値による予測と比べよ。平均する日数と平均絶対誤差には、どんな関係があるかを答えよ。


In [ ]:
# 問1の計算欄：ユーロ・ドル（FRB）の線形トレンド
eur_years = pd.Series((eur_usd.index - eur_usd.index[0]).days / 365.25, index=eur_usd.index, name="経過年数")
eur_starts = ["2000", "2000", "2008", "2017"]
eur_ends = ["2025", "2007", "2016", "2025"]
for k in range(4):
    part = eur_usd.loc[eur_starts[k]:eur_ends[k]]
    model_part = sm.OLS(part, sm.add_constant(eur_years.loc[part.index])).fit()
    print(f"{eur_starts[k]}〜{eur_ends[k]}年：傾き {model_part.params['経過年数']:.4f} ドル／ユーロ／年")

part = eur_usd.loc["2000":"2023"]
model_eur_line = sm.OLS(part, sm.add_constant(eur_years.loc[part.index])).fit()
eur_line = model_eur_line.params["const"] + model_eur_line.params["経過年数"] * eur_years
eur_test = eur_usd.loc["2024"]
print(f"2000〜2023年の線形トレンドを延ばした予測（2024年）：{(eur_test - eur_line.loc['2024']).abs().mean():.4f} ドル／ユーロ")
print(f"今日の値による予測（2024年）：{(eur_test - eur_usd.shift(1).loc['2024']).abs().mean():.4f} ドル／ユーロ")


In [ ]:
# 問2の計算欄：2021〜2023年の線形トレンドを延ばした値
a_short = model_short.params["const"]
b_short = model_short.params["経過年数"]
x_2025 = years.loc["2025-12-30"]
x_2026 = x_2025 + 365 / 365.25  # 2025年12月30日の365日後が2026年12月30日
print(f"2025年12月30日の線形トレンドの値：{a_short + b_short * x_2025:.2f} 円／ドル")
print(f"2026年12月30日の線形トレンドの値：{a_short + b_short * x_2026:.2f} 円／ドル")
print(f"2025年12月30日の実際のレート：{rate.loc['2025-12-30']:.2f} 円／ドル")


In [ ]:
# 追加問題の計算欄


### 演習1の記入欄

- 問1：4つの傾き、2つの平均絶対誤差、次の日の予測に使う方法：
- 問2(a)：2026年12月30日の線形トレンドの値、予測の前提：
- 問2(b)：前提が2025年に成り立ったか：
- 問3（3〜5文）：
- 追加問題：2日、3日、10日の平均絶対誤差、平均する日数との関係：


<details>
<summary>演習1のヒント</summary>

問1では、4つの傾きの符号を見比べよう。2つの平均絶対誤差は、本文の「線形トレンドを延ばした予測」の節と同じように比べる。

問2の(a)では、線形トレンドを延ばすとき、2021〜2023年の傾きがその後どうなると考えているかを書こう。(b)では、2つの値の差を見よう。

問3の①には「移動平均による次の日の予測」の節の平均絶対誤差を使おう。②には、「移動平均」の節の曲線が終わる日と、「後方移動平均」の節の遅れを使おう。

追加問題は、「平均する日数を変えた比較」のセルを計算欄に写し、`[1, 5, 21, 245]` の数を書き換えれば計算できる。今日の値と比べるので、1は残しておこう。各行が何を計算しているかは、その前の表 `steps` と式の部分の表で確かめられる。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

第8回 演習1の解説と演習1の問1で使うユーロ・ドルは [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。
